Live IDS prototype

This notebook prototypes and tests real-time intrusion detection by completing the following:

1. Capture network traffic
2. Convert captured PCAP files to Flow features (CSV)
3. Load the trained ML model
4. Score the flows
5. Generate alerts

Once completed this notebooke will become a standalone py file

In [ ]:
#Comparing columns between the CICIDS2017 and a live sample captured using FlowMeter 

import pandas as pd

cic = pd.read_csv(r"E:\\Project Portfolio\\Dissertation\\Final-Year-IDS\\data\\CICIDS2017\\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
cic.columns = cic.columns.str.strip()

FlowM = pd.read_csv(r"E:\\Project Portfolio\\Dissertation\\Final-Year-IDS\\data\\example.pcap_Flow.csv")
FlowM.columns = FlowM.columns.str.strip()

cic_cols = set(cic.columns)
FlowM_cols = set(FlowM.columns)

print("CIC feature count:", len(cic_cols))
print("Live feature count:", len(FlowM_cols))
print("Shared features:", len(cic_cols & FlowM_cols))

print("\nMissing from live (sample):")
print(list((cic_cols - FlowM_cols))[:20])

print("\nExtra in live (sample):")
print(list((FlowM_cols - cic_cols))[:20])

In [ ]:
# Rename map to allow for easier future allignment and implement a function that will normalise features
# For both data types
rename_map = {
    "Pkt Len Mean": "Packet Length Mean",
    "Pkt Len Std": "Packet Length Std",
    "Pkt Len Max": "Packet Length Max",
    "Pkt Len Min": "Packet Length Min",

    "TotLen Fwd Pkts": "Total Length of Fwd Packets",
    "TotLen Bwd Pkts": "Total Length of Bwd Packets",

    "Tot Fwd Pkts": "Total Fwd Packets",
    "Tot Bwd Pkts": "Total Backward Packets",

    "Fwd Pkt Len Mean": "Fwd Packet Length Mean",
    "Fwd Pkt Len Std": "Fwd Packet Length Std",
    "Fwd Pkt Len Max": "Fwd Packet Length Max",
    "Fwd Pkt Len Min": "Fwd Packet Length Min",

    "PSH Flag Cnt": "PSH Flag Count",
    "ACK Flag Cnt": "ACK Flag Count",
    "RST Flag Cnt": "RST Flag Count",
    "URG Flag Cnt": "URG Flag Count",
    "SYN Flag Cnt": "SYN Flag Count",

    "Init Fwd Win Byts": "Init_Win_bytes_forward",
    "Init Bwd Win Byts": "Init_Win_bytes_backward",

    "Subflow Fwd Pkts": "Subflow Fwd Packets",
    "Subflow Bwd Pkts": "Subflow Bwd Packets",

    "Subflow Fwd Byts": "Subflow Fwd Bytes",
    "Subflow Bwd Byts": "Subflow Bwd Bytes",

    "Bwd Seg Size Avg": "Avg Bwd Segment Size",
    "Fwd Seg Size Avg": "Avg Fwd Segment Size",
}

def normalise_live_features(df, train_features):
    df = df.rename(columns=rename_map)
    df = df.reindex(columns=train_features, fill_value=0)

    return df

In [ ]:
#Loading the previously saved bundel which includes: model, features, threshold, version and creation date

import joblib

bundle_path = r"E:\Project Portfolio\Dissertation\Final-Year-IDS\models\IDS_RF_v1.0"

bundle = joblib.load(bundle_path)
model = bundle["model"]
train_features = bundle["features"]
threshold = float(bundle["threshold"])

print("Loaded Features:", len(train_features))
print("Threshold:", threshold)

In [ ]:
#normalising the live csv sample

X_norm = normalise_live_features(FlowM, train_features)

print("Normalised shape:", X_norm.shape)
print("Columns Identical? ", list(X_norm.columns) == list(train_features))

In [ ]:
# Count how many features ended up all-zero (means missing/unmatched)
all_zero_cols = (X_norm.sum(axis=0) == 0).sum()
print("All-zero columns:", all_zero_cols, "out of", X_norm.shape[1])

# Check for NaN/inf after normalization
import numpy as np
print("Any NaN:", X_norm.isna().any().any())
print("Any inf:", np.isinf(X_norm.to_numpy()).any())

In [ ]:
#testing model predicition

import numpy as np

proba = model.predict_proba(X_norm)[:, 1]
pred = (proba >= threshold).astype(int)

print("Rows:", len(pred))
print("Flagged malicious:", int(pred.sum()))
print("Max proba:", float(np.max(proba)))
print("Mean proba:", float(np.mean(proba)))

In [ ]:
# Function to score the file and return the probability of a malicious packet and the prediction
def score_file(csv_path, model, train_features, threshold):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    X = normalise_live_features(df, train_features)

    proba = model.predict_proba(X)[:,1]
    pred = (proba >= threshold).astype(int)

    results = df.copy()
    results["malicious_prob"] = proba
    results["Prediction"] = pred

    return results

#confirmed safe pcap
sample_file = r"E:\\Project Portfolio\\Dissertation\\Final-Year-IDS\\data\\example.pcap_Flow.csv"

results = score_file(sample_file, model, train_features, threshold)

results.head()

--------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
import os, time, subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

# --- Paths ---
root = Path(r"E:\Project Portfolio\Dissertation\Final-Year-IDS")
pcap_dir = root/"live"/"pcaps"
flow_dir = root/"live"/"flows"
alert_dir = root/"live"/"alerts"
alert_csv = alert_dir/"alerts.csv"

#Iterate through the array and make a the new directory based on the path variables
for d in [pcap_dir, flow_dir, alert_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Model Bundel already loaded (Cell 3)
print("Loaded model. Features:", len(train_features), "Threshold:", threshold)

# CICFlowMeter batch file
cicFlowBat = Path(r"C:\Tools\CICFlowMeter-4.0\bin\cfm.bat")
assert cicFlowBat.exists(), f"Cannot find CICFlowMeter bat: {cicFlowBat}"

In [ ]:
result = subprocess.run(["dumpcap", "-D"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [ ]:
#Select the desired interface (In this case WiFi)
interface = "5"
rotate_time = 10 #pcap duration per file

capture_cmd = [
    "dumpcap",
    "-i", interface,
    "-b", f"duration:{rotate_time}",
    "-w", str(pcap_dir / "capture.pcap")
]

print("Starting capture:", " ".join(capture_cmd))
capture_proc = subprocess.Popen(capture_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
print("Capture PID:", capture_proc.pid)

In [ ]:
# Function to convert pcap to csv using CICFlowMeter
def pcap_to_csv(pcap_path: Path) -> Path:
    output = flow_dir
    cmd = ["cmd", "/c",".\\cfm.bat", str(pcap_path), str(output)]
    completed = subprocess.run(
        cmd,
        cwd=cicFlowBat.parent,   # run inside CICFlowMeter/bin
        capture_output=True,
        text=True
    )

    if completed.returncode != 0:
        raise RuntimeError(
            f"CICFlowMeter failed for {pcap_path.name}\n"
            f"stdout:\n{completed.stdout}\n"
            f"stderr:\n{completed.stderr}"
        )

    if not output.exists():
        raise FileNotFoundError(
            f"Expected CSV not created: {output}\n"
            f"stdout:\n{completed.stdout}\n"
            f"stderr:\n{completed.stderr}"
        )

    return output

In [ ]:
# Function that will score the csv using the trained IDS model and return a dataframe containing flow metadata for readability
# malicious probability, prediction, severity level and an alert message

def score_csv(flow_csv: Path) -> pd.DataFrame:

    # Load and define metadata columns
    df = pd.read_csv(flow_csv)
    df.columns = df.columns.str.strip()

    meta_headers = ["Flow ID", "Src IP", "Dst IP", "Src Port", "Dst Port", "Protocol", "Timestamp"]
    meta_check = [c for c in meta_headers if c in df.columns]
    metadata = df[meta_check].copy()

    # Prepare model features
    X = normalise_live_features(df, train_features)

    # Ensure features are matched
    if list(X.columns) != list(train_features):
        raise ValueError("Feature mismatch after normalisation")

    # Prediction
    chance = model.predict_proba(X)[:, 1]
    prediction = (chance >= threshold).astype(int)

    # Severity level definement
    def severity(p):
        if p > 0.80:
            return "CRITICAL"
        elif p > 0.50:
            return "HIGH"
        elif p > 0.25:
            return "MEDIUM"
        elif p > threshold:
            return "LOW"
        else:
            return "SAFE"
        
    #Alert Messages
    alerts = []

    for i, row in metadata.iterrows():
        if prediction[i] == 1:
            msg = (
                f"⚠️ Potential malicious packet detected | "
                f"{row.get('Src IP', '?')}:{row.get('Src Port', '?')} -> "
                f"{row.get('Dst IP', '?')}:{row.get('Dst Port', '?')} | "
                f"Protocol: {row.get('Protocol', '?')} | "
                f"Probability: {chance[i]:.3f}"
            )
        else:
            msg = ""
        alerts.append(msg)

    # Output Dataframe
    results = metadata.copy()
    results["malicious_prob"] = chance
    results["prediction"] = prediction
    results["severity"] = [severity(p) for p in chance]
    results["alert"] = alerts

    return results


In [ ]:
# Convert pcap to csv file
flow_pcap = Path(r"E:\Project Portfolio\Dissertation\Final-Year-IDS\live\pcaps\capture_00001_20260305165920.pcap")
output_folder = pcap_to_csv(flow_pcap)
print("CICFlowMeter output folder:", output_folder)

#Find the latest CSV file created
csv_file = flow_dir / f"{flow_pcap.stem}.pcap_Flow.csv"
fallback_csv = sorted(flow_dir.glob(f"{flow_pcap.stem}*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)

print("Using CSV:", csv_file)

# Score the csv file using the IDS model
results = score_csv(csv_file)

print(f"Flows analysed: {len(results)}")
print(f"Alerts generated: {int(results['prediction'].sum())}")
print(f"Max probability: {float(results['malicious_prob'].max())}")

alerts = results[results["prediction"] == 1].sort_values("malicious_prob", ascending=False)

display_cols = [c for c in ["Timestamp","Src IP","Src Port","Dst IP","Dst Port","Protocol",
                            "malicious_prob","severity","alert"] if c in results.columns]
display(alerts[display_cols].head(15))

In [ ]:
# -----------------------------
# Configuration
# -----------------------------

interface = "5"          # change if needed
rotate_time = 10         # seconds per capture file
poll_time = 1.0          # loop interval
min_pcap_age = 5.0       # seconds before processing a pcap
top_alerts = 5

alert_log = alert_dir / "alerts_log.csv"

pcap_dir.mkdir(parents=True, exist_ok=True)
flow_dir.mkdir(parents=True, exist_ok=True)
alert_dir.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Helper functions
# -----------------------------

def pcap_old_enough(pcap_path: Path, age: float):
    return (time.time() - pcap_path.stat().st_mtime) >= age


def find_flow_csv(pcap_path: Path):

    matches = sorted(
        flow_dir.glob(f"{pcap_path.stem}*.csv"),
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )

    if not matches:
        raise FileNotFoundError(f"No flow CSV found for {pcap_path.stem}")

    return matches[0]


def log_alert(row):

    df = pd.DataFrame([row])

    if not alert_log.exists():
        df.to_csv(alert_log, index=False)
    else:
        df.to_csv(alert_log, mode="a", header=False, index=False)


# -----------------------------
# Start packet capture
# -----------------------------

capture_cmd = [
    "dumpcap",
    "-i", interface,
    "-b", f"duration:{rotate_time}",
    "-w", str(pcap_dir / "capture.pcap")
]

print("Starting capture:")
print(" ".join(capture_cmd))

capture_proc = subprocess.Popen(
    capture_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print("Capture PID:", capture_proc.pid)
print("Writing PCAP files to:", pcap_dir)


# -----------------------------
# Live IDS loop
# -----------------------------

processed_pcaps = set()

print("\nLive IDS running...")
print("Interrupt the cell to stop.\n")

try:
    while True:
        pcaps = sorted(pcap_dir.glob("*.pcap"), key=lambda p: p.stat().st_mtime)
        for pcap_file in pcaps:

            if pcap_file.name in processed_pcaps:
                continue

            if not pcap_old_enough(pcap_file, min_pcap_age):
                continue

            print(f"\n[PCAP] {pcap_file.name}")

            # Convert PCAP → Flow CSV
            try:
                pcap_to_csv(pcap_file)
                flow_csv = find_flow_csv(pcap_file)
                print("[FLOW]", flow_csv.name)

            except Exception as e:
                print("[FLOW ERROR]", e)
                processed_pcaps.add(pcap_file.name)
                continue

            # Score flows
            try:
                results = score_csv(flow_csv)
                flow_count = len(results)
                alert_count = int(results["prediction"].sum())
                max_prob = float(results["malicious_prob"].max())
                print(f"[SCORE] flows={flow_count} alerts={alert_count} max_prob={max_prob:.3f}")
                alerts = results[results["prediction"] == 1].sort_values(
                    "malicious_prob",
                    ascending=False
                )

                if alert_count > 0:
                    print("[ALERTS]")
                    for alert in alerts["alert"].head(top_alerts):
                        print(alert)
                    top = alerts.iloc[0]

                else:
                    top = None

                log_alert({
                    "time": pd.Timestamp.utcnow().isoformat(),
                    "pcap": pcap_file.name,
                    "flows": flow_count,
                    "alerts": alert_count,
                    "max_prob": max_prob,
                    "src_ip": top.get("Src IP") if top is not None else "",
                    "dst_ip": top.get("Dst IP") if top is not None else "",
                    "severity": top.get("severity") if top is not None else ""
                })

            except Exception as e:
                print("[SCORE ERROR]", e)

            processed_pcaps.add(pcap_file.name)

        time.sleep(poll_time)

except KeyboardInterrupt:
    print("\nStopping IDS...")

finally:
    if capture_proc.poll() is None:
        capture_proc.terminate()

        try:
            capture_proc.wait(timeout=3)
        except subprocess.TimeoutExpired:
            capture_proc.kill()

    print("Capture stopped.")